# 01 — The data, and how we decided what "returned" means

This dataset has **no returns column**. Everything downstream — every rupee figure, every
threshold — rests on a label we had to construct. So this notebook does one job: look hard
at the raw data, and justify the label choice out loud.

Three questions, in order:

1. What is actually in Online Retail II?
2. What is a "return" in a file that only records cancellation invoices?
3. Is the label we built defensible — and where does it leak, distort, or fall short?

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from returnrisk.config import load_config
from returnrisk.plots import use_house_style

use_house_style()
cfg = load_config(Path.cwd().parent / "config.yaml")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("config loaded | seed:", cfg["seed"], "| data source:", cfg["data"]["source"])

config loaded | seed: 42 | data source: uci


## 1. The raw file

One row per invoice line. The first read parses a 45MB workbook; after that it is cached to Parquet.

In [2]:
from returnrisk.data.loader import load_uci, clean

raw = load_uci(cfg)
print(f"{len(raw):,} raw invoice lines")
print(f"{raw['invoice_date'].min():%Y-%m-%d} -> {raw['invoice_date'].max():%Y-%m-%d}")
raw.head()

1,067,371 raw invoice lines
2009-12-01 -> 2011-12-09


,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12.0,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48.0,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24.0,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# What gets dropped, and why. Nothing here is silent.
before = len(raw)
txn = clean(raw, cfg)
print(f"raw lines            {before:>10,}")
print(f"after cleaning       {len(txn):>10,}   ({len(txn)/before:.1%} kept)")
print()
print(f"missing customer id  {raw['customer_id'].isna().sum():>10,}  -> dropped: no history, "
      f"no way to attribute a cancellation")
print(f"cancellation lines   {int(txn['is_cancellation'].sum()):>10,}")
print(f"unique customers     {txn['customer_id'].nunique():>10,}")
print(f"unique stock codes   {txn['stock_code'].nunique():>10,}")

raw lines             1,067,371
after cleaning          820,645   (76.9% kept)

missing customer id     243,007  -> dropped: no history, no way to attribute a cancellation
cancellation lines       17,966
unique customers          5,894
unique stock codes        4,638


### Non-product lines

Postage, bank charges, samples and manual adjustments carry stock codes like `POST`, `M`, `D`.
They are not orderable goods, so leaving them in would manufacture phantom "returns" out of
accounting entries. They are excluded in `clean()`.

In [4]:
admin = raw[raw["stock_code"].astype(str).str.upper().isin(
    ["POST", "D", "DOT", "M", "S", "AMAZONFEE", "BANK CHARGES", "C2", "CRUK", "PADS", "B"])]
print(f"{len(admin):,} non-product lines excluded")
admin["stock_code"].value_counts().head(8)

5,743 non-product lines excluded


stock_code
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
AMAZONFEE         43
Name: count, dtype: int64[pyarrow]

## 2. What a "return" looks like in this file

Cancellations are invoices whose number starts with `C` and whose quantity is negative. That is
the merchant's own record of an order being reversed. There is **no foreign key** back to the
original purchase — so linking is a modelling decision, not a lookup.

In [5]:
cancels = txn[txn["is_cancellation"]]
buys = txn[~txn["is_cancellation"]]
print(f"purchase lines     {len(buys):>10,}")
print(f"cancellation lines {len(cancels):>10,}   ({len(cancels)/len(txn):.2%} of rows)")
cancels[["invoice", "stock_code", "description", "quantity", "invoice_date", "customer_id"]].head()

purchase lines        802,679
cancellation lines     17,966   (2.19% of rows)


,invoice,stock_code,description,quantity,invoice_date,customer_id
175,C489449,22087,PAPER BUNTING WHITE LACE,-12.0,2009-12-01 10:33:00,16321
176,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6.0,2009-12-01 10:33:00,16321
177,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4.0,2009-12-01 10:33:00,16321
178,C489449,21896,POTTING SHED TWINE,-6.0,2009-12-01 10:33:00,16321
179,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12.0,2009-12-01 10:33:00,16321


In [6]:
# How long after a purchase does a cancellation arrive? This is what sets lookahead_days.
merged = cancels.merge(
    buys[["customer_id", "stock_code", "invoice_date"]].rename(columns={"invoice_date": "buy_date"}),
    on=["customer_id", "stock_code"], how="inner",
)
gap = (merged["invoice_date"] - merged["buy_date"]).dt.days
gap = gap[gap >= 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(gap.clip(upper=365), bins=60, color="#2a78d6")
ax.axvline(cfg["label"]["lookahead_days"], color="#d03b3b", linestyle="--", linewidth=2)
ax.annotate(f"lookahead = {cfg['label']['lookahead_days']}d",
            xy=(cfg["label"]["lookahead_days"], ax.get_ylim()[1] * 0.85),
            xytext=(12, 0), textcoords="offset points", color="#d03b3b")
ax.set_xlabel("Days between purchase and cancellation")
ax.set_ylabel("Candidate pairs")
ax.set_title("Most reversals land quickly - 90 days captures the bulk")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

print(f"share of candidate pairs within 90 days: {(gap <= 90).mean():.1%}")
print(gap.describe(percentiles=[.5, .75, .9, .95]).round(1).to_string())

share of candidate pairs within 90 days: 53.6%
count    45896.0
mean       135.1
std        149.5
min          0.0
50%         75.0
75%        214.0
90%        365.0
95%        446.0
max        738.0


**Why 90 days.** The gap distribution is heavily front-loaded — most reversals happen within
weeks. A 90-day window captures the bulk without stretching so far that we lose a large tail of
orders to right-censoring. It is a judgement call, and it lives in `config.yaml` so it can be
challenged: `label.lookahead_days`.

## 3. Building the label

> `returned = 1` if a later cancellation invoice from the same customer reverses at least one
> stock code of the order, within 90 days.

Matching is **most-recent-prior-purchase first**, with quantity consumed greedily so one
returned unit can never mark two orders as returned.

In [7]:
from returnrisk.data.labeling import build_labelled_orders, summarise

orders, report = build_labelled_orders(txn, cfg)
print(summarise(report))
print()
for k, v in report.to_dict().items():
    print(f"  {k:<38} {v}")

Label: returned=1 if a cancellation invoice from the same customer reverses a stock code of the order within 90 days. 14,753/17,966 cancellation lines (82.1%) linked to an originating order. 30,047 orders analysed after dropping 6,573 right-censored orders (placed after 2011-09-10). Base rate = 18.53%.

  label_definition                       returned=1 if a later cancellation invoice from the same customer reverses at least one stock code of the order within 90 days
  n_cancellation_lines                   17966
  n_cancellation_lines_matched           14753
  cancellation_match_rate                0.8212
  n_orders_before_right_censor_drop      36620
  n_orders_analysed                      30047
  n_orders_returned                      5569
  base_rate                              0.185343
  right_censor_cutoff                    2011-09-10 12:50:00
  n_orders_dropped_by_value_filter       6592
  order_value_filter_gbp                 [1.0, 50000.0]


### The three honesty caveats

1. **17.9% of cancellation lines don't link.** The customer has no matching prior purchase of
   that stock code, or it falls outside the window. Those are **dropped**, not reassigned to a
   plausible-looking order.
2. **6,573 orders are right-censored.** Their 90-day window runs past the end of the file, so
   their outcome is unobservable. Keeping them would silently label them `0` and deflate the
   base rate — the single easiest way to accidentally flatter a return model.
3. **This is "a cancellation was recorded", not "goods physically came back."** The data cannot
   distinguish a pre-dispatch cancellation from a post-delivery return.

In [8]:
print(f"orders before censor drop : {report.n_orders_before_censor:>8,}")
print(f"orders analysed           : {report.n_orders_after_censor:>8,}")
print(f"dropped (right-censored)  : {report.n_orders_before_censor - report.n_orders_after_censor:>8,}")
print(f"censor cutoff             : {pd.Timestamp(report.censor_cutoff):%Y-%m-%d}")
print()
print(f"BASE RATE                 : {report.base_rate:.2%}")

orders before censor drop :   36,620
orders analysed           :   30,047
dropped (right-censored)  :    6,573
censor cutoff             : 2011-09-10

BASE RATE                 : 18.53%


## 4. Is the label plausible? Sanity checks against intuition

In [9]:
q = orders.groupby(orders["order_date"].dt.to_period("Q")).agg(
    orders=("returned", "size"), return_rate=("returned", "mean"))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(q)), q["return_rate"], color="#2a78d6", width=0.65)
ax.axhline(orders["returned"].mean(), color="#898781", linestyle="--")
ax.annotate(f"overall {orders['returned'].mean():.1%}", xy=(len(q)-1, orders['returned'].mean()),
            xytext=(0, 6), textcoords="offset points", ha="right", color="#898781")
ax.set_xticks(range(len(q)))
ax.set_xticklabels([str(p) for p in q.index], rotation=0)
ax.set_ylabel("Return rate")
ax.set_title("Return rate is stable across quarters - no structural break")
ax.spines[["top", "right"]].set_visible(False)
plt.show()
q.round(3)

,orders,return_rate
order_date,,
2009Q4,1501,0.177
2010Q1,3594,0.199
2010Q2,4162,0.188
2010Q3,4294,0.187
2010Q4,6066,0.181
2011Q1,3284,0.195
2011Q2,4073,0.178
2011Q3,3073,0.174


In [10]:
# Bigger baskets and bigger orders should return more often. If they didn't, the label is wrong.
orders["value_band"] = pd.cut(orders["order_value_gbp"], [0, 150, 300, 500, 1000, np.inf],
                              labels=["<150", "150-300", "300-500", "500-1k", "1k+"])
orders["lines_band"] = pd.cut(orders["n_lines"], [0, 5, 15, 30, np.inf],
                              labels=["1-5", "6-15", "16-30", "30+"])
print("By order value:")
print(orders.groupby("value_band", observed=True)["returned"].agg(["size", "mean"]).round(3).to_string())
print("\nBy basket size:")
print(orders.groupby("lines_band", observed=True)["returned"].agg(["size", "mean"]).round(3).to_string())

By order value:
            size   mean
value_band             
<150        6897  0.082
150-300     7627  0.146
300-500     8365  0.231
500-1k      4983  0.261
1k+         2175  0.302

By basket size:
            size   mean
lines_band             
1-5         6600  0.101
6-15        8760  0.169
16-30       8619  0.223
30+         6068  0.248


Both move the way intuition says they should — a £1k+ order returns at ~30% against ~8% for a
sub-£150 one. That is weak evidence the label is measuring something real rather than an
artefact of the matching heuristic.

## 5. The features this makes possible — and the one trap

The customer-history features are where return models leak. The trap:

> Customer orders on 5 Jan. That order is returned — but the cancellation lands on **20 March**.
> The customer orders again on **10 March**. At that moment the merchant knows about **zero**
> returns. A naive `prior_return_rate` reports 100% and leaks the future.

So a prior return only counts once its **cancellation date** has passed — not its order date.
The cell below shows the gap this opens: across the book, `prior_returns_observed` is strictly
smaller than the naive "count every past order that was ever returned", and the difference is
exactly the leakage a naive implementation would absorb.

In [11]:
from returnrisk.features import build_feature_frame

feat = build_feature_frame(orders)
print(f"{len(feat):,} orders x {feat.shape[1]} columns")

# Measure the leak a naive implementation would absorb.
# NAIVE: count every earlier order that was EVER returned, regardless of when we found out.
# CORRECT: count only returns whose cancellation date had already passed at checkout.
f = feat.sort_values(["customer_id", "order_date"], kind="stable").copy()
f["naive_prior_returns"] = (
    f.groupby("customer_id")["returned"].transform(lambda s: s.shift(1).cumsum()).fillna(0)
)
gap = f["naive_prior_returns"] - f["prior_returns_observed"]

print(f"\nmean naive prior returns    {f['naive_prior_returns'].mean():.4f}")
print(f"mean OBSERVED prior returns {f['prior_returns_observed'].mean():.4f}")
print(f"rows where they disagree    {(gap > 0).mean():.2%}  "
      f"(these are the rows a naive build would leak on)")
print(f"the naive version is never smaller: {bool((gap >= 0).all())}")

30,047 orders x 36 columns



mean naive prior returns    2.9875
mean OBSERVED prior returns 2.8370
rows where they disagree    9.67%  (these are the rows a naive build would leak on)
the naive version is never smaller: True


In [12]:
# The signal we hope exists: customers who returned before, return again.
band = pd.cut(feat["prior_return_rate"], [-0.01, 0.0, 0.2, 0.4, 1.01],
              labels=["0%", "0-20%", "20-40%", "40%+"])
tbl = feat.groupby(band, observed=True)["returned"].agg(["size", "mean"]).round(3)
tbl.columns = ["orders", "return_rate"]
print("Return rate by the customer's OBSERVED prior return rate:")
print(tbl.to_string())
print(f"\nNew customers (no history): {feat.loc[feat['is_new_customer']==1, 'returned'].mean():.3f} "
      f"on {int((feat['is_new_customer']==1).sum()):,} orders")

Return rate by the customer's OBSERVED prior return rate:
                   orders  return_rate
prior_return_rate                     
0%                  10462        0.127
0-20%                5592        0.137
20-40%               4836        0.234
40%+                 3892        0.372

New customers (no history): 0.171 on 5,265 orders


## Where this leaves us

- **30,047 orders**, base rate **18.53%**, on a real wholesale book.
- A label that is explicit about being a **proxy** (cancellations, not physical returns), with an
  **82.1%** link rate and right-censored orders dropped rather than mislabelled.
- Prior-return history that is **observation-gated** — the single highest-value leakage guard in
  the project, asserted row by row in `tests/test_leakage.py`.

→ `02_train_eval.ipynb` takes this through the temporal split, calibration, and the money layer.